In [11]:
import pandas as pd
import numpy as np

import re

import spacy

import joblib

import warnings
warnings.filterwarnings('ignore')
"""
The Python statement import warnings; warnings.filterwarnings('ignore') serves to suppress or hide all warning 
messages that might otherwise be displayed during the execution of a Python program
"""

"\nThe Python statement import warnings; warnings.filterwarnings('ignore') serves to suppress or hide all warning \nmessages that might otherwise be displayed during the execution of a Python program\n"

## the classes in the TARGET

In [2]:
the_classes={'Bank account or service': 0,
 'Consumer Loan': 1,
 'Credit card': 2,
 'Credit reporting': 3,
 'Debt collection': 4,
 'Mortgage': 5,
 'Student loan': 6}

## the saved models(multinomialNB, Vectorizer)

In [32]:
# importing multinomialNB model from local files

model=joblib.load(r"C:\Users\Aswin\Compliant_NB.joblib")

In [33]:
# importing Vectorizer model from local files

vect=joblib.load(r"C:\Users\Aswin\vectorizer_for_compliant_NB.pkl")

## Preprocessing techniques

In [4]:
def regex_normalize(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower().replace('\n', ' ').replace('\t', ' ') # Lowercase texts and replace /n & /t in the text
    
    text = re.sub(r'\b[0-9/]*X{2,}[0-9/]*\b', '', text) # removes XX/XX/XX19 ,XX/XX/XXXX, XXXX/XXXX/XXXX

    text = re.sub(r'\bx+\b', '', text, flags=re.IGNORECASE) #  replaces xxxx blocks out completely

    text = re.sub(r'\{?\$[0-9.,]+\}?|\{?[0-9.,]+\$\}?', '[MONEY]', text) # replaces any amoount money with currency symbol into token( [MONEY] )
    
    text = re.sub(r'\.{3,}', '', text) # replaces triple dots with nothing in its place

    text = re.sub(r'\d+', '', text) # replaces the numbers like year, two-digit numbers, etc

    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # replaces -(hyphen) before words
    
    text = re.sub(r"\'", "", text) # Removes any backslash that immediately precedes a single quote

    text = re.sub(r'%\s*[0-9.,]+|[0-9.,]+\s*%', '[PERCENTAGE]', text) # replaces the numbers with % with improper placements into token( [PERCENTAGE] )
    
    text = re.sub(r'\s+', ' ', text).strip() # Collapse multiple spaces into a single space

    text = text.replace(r"\'\'", '"').replace(r"\'", "'") # replaces \'' and other slash symbols with single quotaion
    
    return text

In [13]:

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def lemmatize_row(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    
    doc = nlp(text)
    
    # Extract the lemma only if it is NOT punctuation, a symbol, or a standalone period
    lemmas = [token.lemma_ for token in doc if not token.is_punct and token.pos_ != "SYM" and token.lemma_ != "."]
    
    return " ".join(lemmas)

In [15]:
from spacy.lang.en.stop_words import STOP_WORDS


stopwords = set(STOP_WORDS)

custom_stopwords = {"complaint", "case", "number", "regards", "listed", 
                    "attached", "document", "pdf", "company", "bank", 
                    "loan", "citizen", "account","receive", "say", 
                    "tell", "call", "state", "go", "would", "could",
                    "citibank", "citcard", "recieve", "fradulent", 
                    "website", "show", "tell", "say", "talk", "ask",
                    "tell", "say","talk", "advise", "receive", "state", 
                    "include", "question", "website", "show", "transferee",
                    "husband", "father", "law", "attorney", "rep", "supervisor", 
                    "name","numbersince", "forth", "till", "today", 
                    "yesterday", "tomorrow","sentence", "contact", "supply", 
                    "request", "letter", "email", "writing"
                   }

final_stopwords = stopwords.union(custom_stopwords) # Combine them into a single high-speed lookup set


def filter_tokens(text): # FILTER FUNC
    if not isinstance(text, str) or not text.strip():
        return ""
    
    words = text.split()
 
    filtered_words = [word for word in words if word not in final_stopwords and len(word) > 2]
    
    return " ".join(filtered_words)


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=2500,norm='l2',min_df=2,ngram_range=(1, 2))

## Prediction

In [92]:
def predict_complaint(text):
    
    the_classes={'Bank account or service': 0,'Consumer Loan': 1,'Credit card': 2,
                 'Credit reporting': 3,'Debt collection': 4,'Mortgage': 5,'Student loan': 6}

    text = regex_normalize(text) # Remove patterns using regular expression

    text = lemmatize_row(text) # Lemmatization
 
    text = filter_tokens(text) # Remove stopwords 

    text_vector = vect.transform([text]) # Convert to TF-IDF features

    prediction = model.predict(text_vector) # Predicts the class 

    complaint=[k for k , v in the_classes.items() if v == int(prediction[0])] # gets the class name out of a dict

    return complaint[0]

    print(complaint[0])

In [93]:
# text od mortgage complaint

complaint=" Mortgage was transferred to Nationstar as of XXXX/XXXX/XXXX. Since then our payments are not posted in a timely manner or for the amount sent. for example payment cleared our bank on XXXX/XXXX/XXXX for XXXX per the payment history received from Nationstar the payment was not posted until XXXX/XXXX/XXXX and only for amount of XXXX. payment cleared our bank on XXXX/XXXX/XXXX for XXXX but was not posted to our account until XXXX/XXXX/XXXX and only for amount XXXX When our loan was transferred to Nationstar from XXXX we only received a good by letter from XXXX but no welcome package from Nationstar. So had to contact them to make sure the payment amount was the same as XXXX. I was told it was the same so that is what we are sending but they are only posting XXXX Have to call every month to get a payment history as they will not allow use of the website. Try to get information several times as to why it takes so long to post the money and why it is less than we sent. sent them the exact same documents attached and all they will state is they have no record of sending me the history every month and they are posting the money for what we sent. Have been unable to resolve the issue with Nationstar as they are not willing to listen or review the documents I sent to them as proof of what they sent me. Have been calling them every month since XXXX XXXX to get a payment history but they deny sending it each month."

In [94]:
predict_complaint(complaint)

'Mortgage'

In [95]:
# text od debt collection complaint

complaint1="I am writing to request your assistance in looking into the deceptive practices of this collection law-firm above. It appears that they are using tactics that may be violating consumer protection law in debt collection practices depriving consumers of their rights to dispute.1 In XXXX XXXX, I received a notice from the above company, The next day, I contacted their offices as instructed -- the memo dated XXXX/XXXX/XXXX instruct me to contact the plaintiff attorney, not the court. I followed the instructions provided and contacted the plaintiff attorney by phone and also faxed a letter on XXXX/XXXX/XXXX disputing the debt ( see letter ). 2. The company responded with a letter dated XXXX/XXXX/XXXX by sending me a bill with a due date for XXXX. I had requested a bill showing what my balance was back when I made a payment back in XXXX XXXX for {$100.00}. I was disputing the amount owed, disputing the charges. 3. I wrote back to the company and faxed another dispute letter on XXXX XXXX, XXXX continue to dispute the amount owed. 4. The company sent me a response on XXXX/XXXX/XXXX saying that they furnished me the information, I was not disputing that I owe XXXX XXXX, I was disputing that the balance was inaccurate and that I needed proof of the last known charges and activity on the account which was XXXX XXXX. In the last paragraph of their letter it indicated that if I was disputing the amount to send a letter and so on.. 5. On XXXX XXXX, XXXX -- I sent another letter to the company disputing the balance and requesting the documents again. The company never responded to my XXXX XXXX letter, they since had not communication with me, until XXXX XXXX XXXX when I received a letter from them with a copy of a DEFAULT judgment that they filed the court clerk 's office indicating that I failed to respond to their judgment. Facts:1. The only judgment that the above firm served me with was the original judgment which I am attaching dispute letters showing that I RESPONDED as instructed to their office on XXXX different occasions. 2. The plaintiff failed to respond to my dispute and furnish the information provided-and was probably unable to obtain proof of the original of the debt3. Instead of using credible legal procedure to settle the debts, they utilized unfaithful and dirty tactics, violated my rights. 4. Went to the court, committed perjury under the law by filing false documents with the court that I defaulted on the judgment and failed to respond when in fact I responded and they failed to furnish proof. 5. I went the court house and the clerk 's office, I was told that the company did not notify their offices that they had been in contact with me, instead they the company told the court and clerk 's office that I did not respond to their summons the the clerks office granted them the default judgement based on their false information that I did not respond to their summons. They filed a false affirmation with the clerk 's office."

In [96]:
predict_complaint(complaint1)

'Debt collection'

## transformers

In [63]:
# transformers for finding the emotion and severity of the text

from transformers import pipeline

emotion_classifier = pipeline("text-classification",model="j-hartmann/emotion-english-distilroberta-base",top_k=None)
polarity_classifier = pipeline("sentiment-analysis",model="cardiffnlp/twitter-roberta-base-sentiment-latest")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [90]:
complaint

' Mortgage was transferred to Nationstar as of XXXX/XXXX/XXXX. Since then our payments are not posted in a timely manner or for the amount sent. for example payment cleared our bank on XXXX/XXXX/XXXX for XXXX per the payment history received from Nationstar the payment was not posted until XXXX/XXXX/XXXX and only for amount of XXXX. payment cleared our bank on XXXX/XXXX/XXXX for XXXX but was not posted to our account until XXXX/XXXX/XXXX and only for amount XXXX When our loan was transferred to Nationstar from XXXX we only received a good by letter from XXXX but no welcome package from Nationstar. So had to contact them to make sure the payment amount was the same as XXXX. I was told it was the same so that is what we are sending but they are only posting XXXX Have to call every month to get a payment history as they will not allow use of the website. Try to get information several times as to why it takes so long to post the money and why it is less than we sent. sent them the exact s

In [107]:
def analyze_complaint(complaint):
    
    category = predict_complaint(complaint) # Predicts the class
    
    emotion = emotion_classifier(text)[0] # Emotion model

    emotion = max(emotion, key=lambda x: x["score"]) # gets maximum score from it

    polarity = polarity_classifier(text)[0] # model finds the polarity score of the complaint

    return {
        "category": category,
        "emotion": emotion["label"],
        "emotion_score": emotion["score"],
        "sentiment": polarity["label"],
        "severity_score": polarity["score"]
    }

In [108]:
res=analyze_complaint(complaint)

In [110]:
res

{'category': 'Mortgage',
 'emotion': 'anger',
 'emotion_score': 0.9724326133728027,
 'sentiment': 'negative',
 'severity_score': 0.9243983626365662}